**Milestone-1**
**Module 1: Scraping Setup and Execution **

**Identify metrics to scrape (as per doc)**
Required (minimum):
Country
power_index_rank
total_manpower
active_personnel
reserve_personnel
total_aircraft
fighter_aircraft
naval_assets
tanks
defense_budget
gdp"
**Takeaway:**
These metrics collectively represent:

🪖 Human strength (manpower)

✈️ Air capability

🚢 Naval capability

🛡️ Land power

💰 Economic strength & military funding

📊 Overall global ranking

These fields are the minimum required dataset columns for Task-1 and must be present in the scraped dataset.

**Define Raw Data Schema**

The raw data schema defines how the scraped data will be structured before cleaning and processing.
 Column Name        Data Type  Description                  
 country            string     Name of the country          
 power_index_rank   integer    Global military rank         
 total_manpower     integer    Total available manpower     
 active_personnel   integer    Active military personnel    
 reserve_personnel  integer    Reserve military personnel   
 total_aircraft     integer    Total number of aircraft     
 fighter_aircraft   integer    Total fighter jets           
 naval_assets       integer    Total naval ships/assets     
 tanks              integer    Total battle tanks           
 defense_budget     float      Annual defense budget (USD)  
 gdp                float      Gross Domestic Product (USD) 


In [34]:
#sample code
import pandas as pd
import json

# Sample dataset
data = [
    ["India", 4, 66238168, 1450000, 1155000, 2229, 606, 293, 4614, 75000000000, 3730000000000]
]

columns = [
    "country",
    "power_index_rank",
    "total_manpower",
    "active_personnel",
    "reserve_personnel",
    "total_aircraft",
    "fighter_aircraft",
    "naval_assets",
    "tanks",
    "defense_budget",
    "gdp"
]

df = pd.DataFrame(data, columns=columns)
india_data = df[df["country"] == "India"].iloc[0].to_dict()

json_output = json.dumps(india_data, indent=2)

print(json_output)

{
  "country": "India",
  "power_index_rank": 4,
  "total_manpower": 66238168,
  "active_personnel": 1450000,
  "reserve_personnel": 1155000,
  "total_aircraft": 2229,
  "fighter_aircraft": 606,
  "naval_assets": 293,
  "tanks": 4614,
  "defense_budget": 75000000000,
  "gdp": 3730000000000
}


**Scrape data from one country page correctly. use this link to scrape data**


In [30]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd
import os

# -----------------------------
# INDIA URL
# -----------------------------
url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id=india"

# -----------------------------
# HEADERS (Prevents 403 error)
# -----------------------------
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

# -----------------------------
# FETCH PAGE
# -----------------------------
response = requests.get(url, headers=headers, timeout=15)
response.raise_for_status()

# Use html.parser (safer if lxml not installed)
soup = BeautifulSoup(response.text, "html.parser")

data = {}

# -----------------------------
# COUNTRY NAME
# -----------------------------
title = soup.find("h1")
if title:
    data["country"] = title.get_text(strip=True)
else:
    data["country"] = "India"

# -----------------------------
# POWER INDEX EXTRACTION
# -----------------------------
overview_text = soup.find("span", class_="textNormal")

if overview_text:
    text = overview_text.get_text(" ", strip=True)

    rank_match = re.search(r"ranked\s+(\d+)", text)
    score_match = re.search(r"score of\s+([\d.]+)", text)

    data["power_index_rank"] = rank_match.group(1) if rank_match else None
    data["power_index_score"] = score_match.group(1) if score_match else None
else:
    data["power_index_rank"] = None
    data["power_index_score"] = None

# -----------------------------
# METRIC EXTRACTION
# -----------------------------
metric_blocks = soup.find_all("div", class_="specsGenContainers")

for block in metric_blocks:
    label = block.find("span", class_="textLarge")
    value = block.find("span", class_="textWhite")

    if label and value:
        key = (
            label.get_text(strip=True)
            .replace(":", "")
            .lower()
            .replace(" ", "_")
        )
        data[key] = value.get_text(strip=True)

# -----------------------------
# PRINT OUTPUT
# -----------------------------
print("India Data Extracted:\n")
for k, v in data.items():
    print(f"{k}: {v}")

# -----------------------------
# SAVE CSV (Ensure folder exists)
# -----------------------------
os.makedirs("data", exist_ok=True)

df = pd.DataFrame([data])
df.to_csv("data/india_test.csv", index=False)

print("\nSaved to data/india_test.csv")

India Data Extracted:

country: 2026India Military Strength
power_index_rank: 4
power_index_score: 0.1346
purchasing_power_parity: 3
foreign_exchange/gold: 5
defense_budget: 5
external_debt: 110
square_land_area: 7
coastline_coverage: 91
shared_borders: 129
waterways_(usable): 10
total_population: 2
available_manpower: 2
fit-for-service: 2
reaching_mil_age_annually: 1
tot_mil._personnel_(est.): 4,958,000
active_personnel: 2
reserve_personnel: 8
paramilitary: 2
air_force_personnel*: 3
army_personnel*: 3
navy_personnel*: 4
yearly_mobilization_potential: 23,652,761
aircraft_total: 4
fighters: 4
attack_types: 4
transports_(fixed-wing): 4
trainers: 8
special-mission: 5
tanker_fleet: 11
helicopters: 4
attack_helicopters: 9
tanks: 5
vehicles: 2
self-propelled_artillery: 35
towed_artillery: 3
mlrs_(rocket_artillery): 16
total_assets: 4
total_tonnage: 5
aircraft_carriers: 3
helicopter_carriers: 145
destroyers: 5
frigates: 3
corvettes: 5
submarines: 7
patrol_vessels: 7
mine_warfare: 145
oil_prod

**Takeaway**

This scraping script provides a starting template to pull structured military stats directly from Global Firepower’s country detail page. You’ll need to refine CSS selectors according to the actual HTML structure of that page to ensure accurate extraction.

**Automate Scraping Using Provided URL List**

In [31]:
# Step 1: Generate full URLs
base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="

country = [
    "afghanistan","albania","algeria","angola","argentina","armenia",
    "australia","austria","azerbaijan","bahrain","bangladesh","belarus",
    "belgium","bhutan","bolivia","bosnia-and-herzegovina","botswana",
    "brazil","bulgaria","burkina-faso","burma","cabo-verde","cambodia",
    "cameroon","canada","central-african-republic","chad","chile",
    "china","colombia","comoros","croatia","cuba","czech-republic",
    "denmark","djibouti","dominican-republic","ecuador","egypt",
    "el-salvador","equatorial-guinea","eritrea","estonia","ethiopia",
    "finland","france","gabon","georgia","germany","ghana","greece",
    "guatemala","honduras","hungary","india","indonesia","iran","iraq",
    "ireland","israel","italy","ivory-coast","jamaica","japan","jordan",
    "kazakhstan","kenya","kosovo","kuwait","kyrgyzstan","laos","latvia",
    "lebanon","libya","lithuania","madagascar","malaysia","mali",
    "mauritania","mexico","moldova","mongolia","montenegro","morocco",
    "mozambique","namibia","nepal","netherlands","new-zealand",
    "nicaragua","niger","nigeria","north-korea","north-macedonia",
    "norway","oman","pakistan","palestine","panama","paraguay","peru",
    "philippines","poland","portugal","qatar","romania","russia",
    "sao-tome-and-principe","saudi-arabia","senegal","serbia",
    "seychelles","singapore","slovakia","slovenia","somalia",
    "south-africa","south-korea","south-sudan","spain","sri-lanka",
    "sudan","sweden","switzerland","syria","taiwan","tajikistan",
    "tanzania","thailand","timor-leste","tunisia","turkey",
    "turkmenistan","uganda","ukraine","united-arab-emirates",
    "united-kingdom","united-states","uruguay","uzbekistan",
    "venezuela","vietnam","yemen","zambia","zimbabwe"
]

# Automatically generate full URLs
urls = [f"{base_url}{cid}" for cid in country]

save_path = "data"
os.makedirs(save_path, exist_ok=True)

url_file = os.path.join(save_path, "all_country_urls.txt")

with open(url_file, "w") as f:
    for url in urls:
        f.write(url + "\n")

print("URLs saved at:", url_file)



URLs saved at: data\all_country_urls.txt


In [32]:
from bs4 import BeautifulSoup
import re
import pandas as pd
import os
import requests
import time

# =============================================
# SCRAPE ALL 145 COUNTRIES DATA
# =============================================

base_url = "https://www.globalfirepower.com/country-military-strength-detail.php?country_id="

countries = [
    "afghanistan","albania","algeria","angola","argentina","armenia",
    "australia","austria","azerbaijan","bahrain","bangladesh","belarus",
    "belgium","bhutan","bolivia","bosnia-and-herzegovina","botswana",
    "brazil","bulgaria","burkina-faso","burma","cabo-verde","cambodia",
    "cameroon","canada","central-african-republic","chad","chile",
    "china","colombia","comoros","croatia","cuba","czech-republic",
    "denmark","djibouti","dominican-republic","ecuador","egypt",
    "el-salvador","equatorial-guinea","eritrea","estonia","ethiopia",
    "finland","france","gabon","georgia","germany","ghana","greece",
    "guatemala","honduras","hungary","india","indonesia","iran","iraq",
    "ireland","israel","italy","ivory-coast","jamaica","japan","jordan",
    "kazakhstan","kenya","kosovo","kuwait","kyrgyzstan","laos","latvia",
    "lebanon","libya","lithuania","madagascar","malaysia","mali",
    "mauritania","mexico","moldova","mongolia","montenegro","morocco",
    "mozambique","namibia","nepal","netherlands","new-zealand",
    "nicaragua","niger","nigeria","north-korea","north-macedonia",
    "norway","oman","pakistan","palestine","panama","paraguay","peru",
    "philippines","poland","portugal","qatar","romania","russia",
    "sao-tome-and-principe","saudi-arabia","senegal","serbia",
    "seychelles","singapore","slovakia","slovenia","somalia",
    "south-africa","south-korea","south-sudan","spain","sri-lanka",
    "sudan","sweden","switzerland","syria","taiwan","tajikistan",
    "tanzania","thailand","timor-leste","tunisia","turkey",
    "turkmenistan","uganda","ukraine","united-arab-emirates",
    "united-kingdom","united-states","uruguay","uzbekistan",
    "venezuela","vietnam","yemen","zambia","zimbabwe"
]

urls = [f"{base_url}{cid}" for cid in countries]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

all_data = []

print(f"Starting scrape for {len(urls)} countries...\n")

for idx, url in enumerate(urls, 1):
    try:
        response = requests.get(url, headers=headers, timeout=15)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, "html.parser")
        
        data = {}
        
        # COUNTRY NAME
        title = soup.find("h1")
        if title:
            data["country"] = title.get_text(strip=True).replace("2026 ", "").replace(" Military Strength", "")
        else:
            country_id = url.split("=")[-1]
            data["country"] = country_id.replace("-", " ").title()
        
        # POWER INDEX EXTRACTION
        overview_text = soup.find("span", class_="textNormal")
        if overview_text:
            text = overview_text.get_text(" ", strip=True)
            rank_match = re.search(r"ranked\s+(\d+)", text)
            score_match = re.search(r"score of\s+([\d.]+)", text)
            data["power_index_rank"] = int(rank_match.group(1)) if rank_match else None
            data["power_index_score"] = float(score_match.group(1)) if score_match else None
        else:
            data["power_index_rank"] = None
            data["power_index_score"] = None
        
        # METRIC EXTRACTION
        metric_blocks = soup.find_all("div", class_="specsGenContainers")
        for block in metric_blocks:
            label = block.find("span", class_="textLarge")
            value = block.find("span", class_="textWhite")
            if label and value:
                key = (
                    label.get_text(strip=True)
                    .replace(":", "")
                    .lower()
                    .replace(" ", "_")
                )
                raw_value = value.get_text(strip=True)
                
                # Convert to appropriate types
                if key in ["total_manpower", "active_personnel", "reserve_personnel", "total_aircraft", "fighter_aircraft", "naval_assets", "tanks"]:
                    try:
                        data[key] = int(raw_value.replace(",", "").replace("+", ""))
                    except (ValueError, AttributeError):
                        data[key] = None
                elif key in ["defense_budget", "gdp"]:
                    try:
                        # Remove special characters and convert to float
                        cleaned = raw_value.replace(",", "").replace("$", "").replace(" ", "").replace("billion", "e9").replace("million", "e6")
                        data[key] = float(cleaned) if cleaned else None
                    except (ValueError, AttributeError):
                        data[key] = None
                else:
                    data[key] = raw_value
        
        all_data.append(data)
        print(f"[{idx}/{len(urls)}] ✓ {data['country']}")
        
        # Add delay to avoid being blocked
        time.sleep(1)
    
    except Exception as e:
        print(f"[{idx}/{len(urls)}] ✗ Error scraping {url}: {str(e)}")
        continue

print(f"\n{'='*60}")
print(f"Total countries scraped: {len(all_data)}")
print(f"{'='*60}\n")

# =============================================
# CREATE RAW CSV FILE WITH REQUIRED SCHEMA
# =============================================

# Define required columns according to schema
required_columns = [
    "country",
    "power_index_rank",
    "total_manpower",
    "active_personnel",
    "reserve_personnel",
    "total_aircraft",
    "fighter_aircraft",
    "naval_assets",
    "tanks",
    "defense_budget",
    "gdp"
]

# Create DataFrame
df = pd.DataFrame(all_data)

# Create a copy for raw data
df_raw = df.copy()

# Add missing columns with None values
for col in required_columns:
    if col not in df_raw.columns:
        df_raw[col] = None

# Select and reorder columns according to schema
df_raw = df_raw[required_columns]

# Handle data type conversions
numeric_cols = ["power_index_rank", "total_manpower", "active_personnel", "reserve_personnel", 
                "total_aircraft", "fighter_aircraft", "naval_assets", "tanks"]
float_cols = ["defense_budget", "gdp"]

for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

for col in float_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# Define output path
raw_data_path = r"C:\Users\K LASYA SREE\OneDrive\Desktop\23071A0494\Milatory dashboard\Militar_raw_data.csv"

# Create directory if it doesn't exist
os.makedirs(os.path.dirname(raw_data_path), exist_ok=True)

# Save raw CSV file
df_raw.to_csv(raw_data_path, index=False)

print(f"✓ Raw CSV file created successfully!")
print(f"Path: {raw_data_path}")
print(f"Total records: {len(df_raw)}")
print(f"Total columns: {len(required_columns)}")
print(f"\nSchema columns:\n{list(df_raw.columns)}")
print(f"\nData types:\n{df_raw.dtypes}")
print(f"\nFirst 5 rows:\n{df_raw.head()}")
print(f"\nLast 5 rows:\n{df_raw.tail()}")
print(f"\nMissing values:\n{df_raw.isnull().sum()}")


Starting scrape for 145 countries...

[1/145] ✓ 2026Afghanistan
[2/145] ✓ 2026Albania
[3/145] ✓ 2026Algeria
[4/145] ✓ 2026Angola
[5/145] ✓ 2026Argentina
[6/145] ✓ 2026Armenia
[7/145] ✓ 2026Australia
[8/145] ✓ 2026Austria
[9/145] ✓ 2026Azerbaijan
[10/145] ✓ 2026Bahrain
[11/145] ✓ 2026Bangladesh
[12/145] ✓ 2026Belarus
[13/145] ✓ 2026Belgium
[14/145] ✓ 2026Bhutan
[15/145] ✓ 2026Bolivia
[16/145] ✓ 2026Bosnia and Herzegovina
[17/145] ✓ 2026Botswana
[18/145] ✓ 2026Brazil
[19/145] ✓ 2026Bulgaria
[20/145] ✓ 2026Burkina Faso
[21/145] ✓ 2026Afghanistan
[22/145] ✓ 2026Afghanistan
[23/145] ✓ 2026Cambodia
[24/145] ✓ 2026Cameroon
[25/145] ✓ 2026Canada
[26/145] ✓ 2026Central African Republic
[27/145] ✓ 2026Chad
[28/145] ✓ 2026Chile
[29/145] ✓ 2026China
[30/145] ✓ 2026Colombia
[31/145] ✓ 2026Afghanistan
[32/145] ✓ 2026Croatia
[33/145] ✓ 2026Cuba
[34/145] ✓ 2026Czechia
[35/145] ✓ 2026Denmark
[36/145] ✓ 2026Afghanistan
[37/145] ✓ 2026Dominican Republic
[38/145] ✓ 2026Ecuador
[39/145] ✓ 2026Egypt
[40/145

Takeaway:
 Same scraping logic
 Fully automated
 Scalable to 100+ countries
 Raw dataset ready
 Next step → Cleaning & Missing Value Handling

**Export raw data**
C:\Users\K LASYA SREE\OneDrive\Desktop\23071A0494\Milatory dashboard\Militar_raw_data.csv

In [33]:
import pandas as pd

In [ ]:
# Load the raw data CSV
raw_data_path = r"C:\Users\K LASYA SREE\OneDrive\Desktop\23071A0494\Milatory dashboard\Militar_raw_data.csv"
df_raw = pd.read_csv(raw_data_path)


**Module 2 : Data Cleaning & Standardization**

Target:Convert noisy scraped data into a clean, numeric, analytics-ready dataset."


STEP-1 : LOAD RAW DATA


In [35]:
import pandas as pd
import os

print("="*80)
print("STEP 1: LOAD RAW DATA & BASIC ANALYSIS")
print("="*80)

# Load previous day's raw file
file_path = "military_power_data_complete.csv"   # change if needed
df = pd.read_csv(file_path)

print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nBasic Statistics:")
print(df.describe())

print(f"\nTotal Rows: {len(df)}")
print(f"Total Columns: {len(df.columns)}")

STEP 1: LOAD RAW DATA & BASIC ANALYSIS

First 5 rows:
         country  power_index_rank  total_manpower  tanks  total_aircraft  \
0         Russia                 1         1091570   9026            3042   
1          China                 2         1253586   8857            3861   
2  United States                 3         1304796   8060            4624   
3          India                 4          798457   3595             873   
4          Japan                 4          522873   3358            1045   

   fighter_aircraft  naval_assets_total  defense_budget_millions  gdp_millions  
0              1467                 418                   637122      21907142  
1              1742                 436                   672551      22924003  
2              2284                 455                   620486      23147612  
3               365                 108                    51696       3315409  
4               424                 162                    50960       3956684

**Takeaway:**
Raw data is ready for cleaning


In [36]:
#step 2 : Identify data noise
print("\n" + "="*80)
print("STEP 2: IDENTIFY DATA NOISE")
print("="*80)

# Check duplicates
duplicates = df.duplicated().sum()
print("Duplicate rows:", duplicates)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check negative values in numeric columns
numeric_cols = df.select_dtypes(include=['int64','float64']).columns

for col in numeric_cols:
    negative_count = (df[col] < 0).sum()
    if negative_count > 0:
        print(f"Negative values found in {col}: {negative_count}")


STEP 2: IDENTIFY DATA NOISE
Duplicate rows: 0

Missing Values:
country                    0
power_index_rank           0
total_manpower             0
tanks                      0
total_aircraft             0
fighter_aircraft           0
naval_assets_total         0
defense_budget_millions    0
gdp_millions               0
dtype: int64


Takeaway: no data noise

In [37]:
#step 3 : Clean numeric columns
print("\n" + "="*80)
print("STEP 3: CLEAN NUMERIC COLUMNS")
print("="*80)

# Remove commas if present and convert to numeric
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Remove impossible values (example rule)
df['fighter_aircraft'] = df.apply(
    lambda row: min(row['fighter_aircraft'], row['total_aircraft']),
    axis=1
)

print("Numeric cleaning complete.")


STEP 3: CLEAN NUMERIC COLUMNS
Numeric cleaning complete.


Takeaway :clean numeric columns completed

In [38]:
# step 4 : Standardize Column Names
print("\n" + "="*80)
print("STEP 4: STANDARDIZE COLUMN NAMES")
print("="*80)

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print("Updated Columns:")
print(df.columns.tolist())


STEP 4: STANDARDIZE COLUMN NAMES
Updated Columns:
['country', 'power_index_rank', 'total_manpower', 'tanks', 'total_aircraft', 'fighter_aircraft', 'naval_assets_total', 'defense_budget_millions', 'gdp_millions']


Takeaway :Standardized and updationn of column names completed

#step 5 : Handle Missing Values
print("\n" + "="*80)
print("STEP 5: HANDLE MISSING VALUES")
print("="*80)

# Fill numeric missing values with median
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Drop rows if country name missing
df.dropna(subset=['country'], inplace=True)

print("Missing value treatment complete.")
print(df.isnull().sum())

Takeaway: This step includes taht there is no missing value

In [40]:
#step 6: Validate Cleaned Data
print("\n" + "="*80)
print("STEP 6: VALIDATE CLEANED DATA")
print("="*80)

# Check duplicates again
print("Duplicate rows after cleaning:", df.duplicated().sum())

# Check fighter aircraft <= total aircraft
invalid_rows = df[df['fighter_aircraft'] > df['total_aircraft']]
print("Invalid fighter > total aircraft rows:", len(invalid_rows))

print("\nFinal Dataset Shape:", df.shape)
print("\nFinal Summary Statistics:")
print(df.describe())


STEP 6: VALIDATE CLEANED DATA
Duplicate rows after cleaning: 0
Invalid fighter > total aircraft rows: 0

Final Dataset Shape: (145, 9)

Final Summary Statistics:
       power_index_rank  total_manpower        tanks  total_aircraft  \
count        145.000000    1.450000e+02   145.000000      145.000000   
mean          80.282759    1.172531e+05   613.910345      251.737931   
std           34.885070    2.255171e+05  1340.443749      616.330843   
min            1.000000    4.690000e+02     3.000000        7.000000   
25%           59.000000    2.358100e+04   156.000000       55.000000   
50%           84.000000    5.048100e+04   284.000000       96.000000   
75%          106.000000    8.367400e+04   427.000000      126.000000   
max          145.000000    1.304796e+06  9026.000000     4624.000000   

       fighter_aircraft  naval_assets_total  defense_budget_millions  \
count        145.000000          145.000000               145.000000   
mean         104.531034           46.765517 

Takeaway:Cleaned data set is validated

In [41]:
#Step 7: Export Cleaned Dataset
print("\n" + "="*80)
print("STEP 7: EXPORT CLEANED DATASET")
print("="*80)

clean_path = "military_power_data_cleaned.csv"
df.to_csv(clean_path, index=False)

print("Cleaned dataset saved at:")
print(os.path.abspath(clean_path))


STEP 7: EXPORT CLEANED DATASET
Cleaned dataset saved at:
c:\Users\K LASYA SREE\OneDrive\Desktop\23071A0494\Milatory dashboard\military_power_data_cleaned.csv


Takeaway:Data set is exported

**TAsk7: KPI Engineering & Feature Creation**
TArget: Create analytical KPIs from cleaned data to support all 4 dashboards.

In [42]:
#step 1: Load Clean Dataset
import pandas as pd
import os

print("="*80)
print("STEP 1: LOAD CLEAN DATASET")
print("="*80)

file_path = "military_power_data_cleaned.csv"
df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print(df.head())

STEP 1: LOAD CLEAN DATASET
Dataset Shape: (145, 9)
         country  power_index_rank  total_manpower  tanks  total_aircraft  \
0         Russia                 1         1091570   9026            3042   
1          China                 2         1253586   8857            3861   
2  United States                 3         1304796   8060            4624   
3          India                 4          798457   3595             873   
4          Japan                 4          522873   3358            1045   

   fighter_aircraft  naval_assets_total  defense_budget_millions  gdp_millions  
0              1467                 418                   637122      21907142  
1              1742                 436                   672551      22924003  
2              2284                 455                   620486      23147612  
3               365                 108                    51696       3315409  
4               424                 162                    50960       3956684  


Takeaway: cleaned dataset is loaded

Step 2: Define Required KPIs

We will create the following analytical KPIs:
tank_density =	tanks / total_manpower
aircraft_per_100k =	(total_aircraft / total_manpower) * 100000
fighter_ratio	= fighter_aircraft / total_aircraft
naval_strength_index =	naval_assets_total / total_manpower
defense_budget_per_soldier	= defense_budget_millions / total_manpower
gdp_to_defense_ratio =	gdp_millions / defense_budget_millions
composite_power_score =	weighted military strength score

In [43]:
#Step 3: KPI Calculations
print("\n" + "="*80)
print("STEP 3: KPI CALCULATIONS")
print("="*80)

# Avoid division by zero
df.replace(0, 1, inplace=True)

df["tank_density"] = df["tanks"] / df["total_manpower"]

df["aircraft_per_100k"] = (
    df["total_aircraft"] / df["total_manpower"]
) * 100000

df["fighter_ratio"] = (
    df["fighter_aircraft"] / df["total_aircraft"]
)

df["naval_strength_index"] = (
    df["naval_assets_total"] / df["total_manpower"]
)

df["defense_budget_per_soldier"] = (
    df["defense_budget_millions"] / df["total_manpower"]
)

df["gdp_to_defense_ratio"] = (
    df["gdp_millions"] / df["defense_budget_millions"]
)

# Composite Power Score (Weighted Model)
df["composite_power_score"] = (
    df["tanks"] * 0.25 +
    df["total_aircraft"] * 0.25 +
    df["naval_assets_total"] * 0.20 +
    df["total_manpower"] * 0.20 +
    df["defense_budget_millions"] * 0.10
)

print("KPI creation complete.")


STEP 3: KPI CALCULATIONS
KPI creation complete.


Takeaway: Kpi creation is completed


In [45]:
#Step 4: Metadata Enrichment



print("\n" + "="*80)
print("STEP 4: METADATA ENRICHMENT")
print("="*80)

# Military Tier Classification
def classify_power(rank):
    if rank <= 10:
        return "Super Power"
    elif rank <= 50:
        return "Major Power"
    elif rank <= 100:
        return "Developing Military"
    else:
        return "Emerging Military"

df["military_tier"] = df["power_index_rank"].apply(classify_power)

# Budget Strength Category
df["budget_category"] = pd.qcut(
    df["defense_budget_millions"],
    q=4,
    labels=["Low", "Medium", "High", "Very High"]
)

print("Metadata enrichment complete.")


STEP 4: METADATA ENRICHMENT
Metadata enrichment complete.


Tkeaway: this medadata enrichment is completed


In [46]:
#Validate KPI Values
print("\n" + "="*80)
print("STEP 5: VALIDATE KPI VALUES")
print("="*80)

# Check for NaN
print("Missing KPI values:")
print(df.isnull().sum())

# Check extreme values
print("\nKPI Summary Statistics:")
print(df[[
    "tank_density",
    "aircraft_per_100k",
    "fighter_ratio",
    "naval_strength_index",
    "defense_budget_per_soldier",
    "gdp_to_defense_ratio",
    "composite_power_score"
]].describe())


STEP 5: VALIDATE KPI VALUES
Missing KPI values:
country                       0
power_index_rank              0
total_manpower                0
tanks                         0
total_aircraft                0
fighter_aircraft              0
naval_assets_total            0
defense_budget_millions       0
gdp_millions                  0
tank_density                  0
aircraft_per_100k             0
fighter_ratio                 0
naval_strength_index          0
defense_budget_per_soldier    0
gdp_to_defense_ratio          0
composite_power_score         0
military_tier                 0
budget_category               0
dtype: int64

KPI Summary Statistics:
       tank_density  aircraft_per_100k  fighter_ratio  naval_strength_index  \
count    145.000000         145.000000     145.000000            145.000000   
mean       0.008729         346.864963       0.303741              0.000945   
std        0.011234         487.169466       0.086510              0.001127   
min        0.000669  

Takeaway: Kpi values are validated

In [ ]:
#Step 6:Export Final Dataset
print("\n" + "="*80)
print("STEP 6: EXPORT FINAL DATASET")
print("="*80)

final_path = "military_power_kpi_dataset.csv"
df.to_csv(final_path, index=False)

print("Final KPI dataset saved at:")
print(os.path.abspath(final_path))